# LLM Adaptation Workflow
### Notebook 01b — Neural Networks from Scratch

---

## What this notebook does

Every model in this project — the LLM, the embedding model, the reward model — is a neural network. This notebook builds one from scratch so you understand exactly what is happening inside them.

We'll go in four stages:

1. **A single neuron** — the basic building block
2. **The loss function** — visualising what the network is trying to minimise
3. **Gradient descent in action** — watching the optimiser navigate the loss surface
4. **A feedforward network** — layers of neurons trained on financial sentiment
5. **A tiny language model** — trained character-by-character on financial text, the same principle as GPT

By the end, when you see `model.generate(...)` in Notebook 04, you'll know exactly what computation is happening under the hood.

We use **raw PyTorch only** — no HuggingFace, no abstractions.

---

## The core idea

A neural network is a function that maps numbers to numbers.

```
input numbers  →  [layers of maths]  →  output numbers
```

The "maths" is just: multiply by some weights, add a bias, apply a non-linearity. Repeat many times. The weights start random. Training adjusts them until the output is useful.

That's it. Everything else — transformers, attention, RLHF — is built on top of this idea.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---

## Part 1 — A single neuron

A neuron takes several numbers as input, multiplies each by a learned weight, sums them up, adds a bias, and applies an activation function.

```
inputs:   x1=0.5,  x2=0.3,  x3=0.9
weights:  w1=0.2,  w2=-0.5, w3=0.8
bias:     b=0.1

output = activation( x1*w1 + x2*w2 + x3*w3 + b )
       = activation( 0.5×0.2 + 0.3×(-0.5) + 0.9×0.8 + 0.1 )
       = activation( 0.77 )
       = ReLU(0.77) = 0.77
```

The weights and bias are the things training will adjust.

In [ ]:
x = torch.tensor([0.5, 0.3, 0.9])
w = torch.tensor([0.2, -0.5, 0.8])
b = torch.tensor(0.1)

z = torch.dot(x, w) + b
output = F.relu(z)

print(f"Linear combination z = {z.item():.4f}")
print(f"After ReLU:   output = {output.item():.4f}")
print()
print("Why activation functions matter:")
print("  Without them, stacking layers = just one big linear transformation.")
print("  With them, we can learn any non-linear pattern.")

In [ ]:
# Visualise the four most common activation functions
x_range = torch.linspace(-3, 3, 200)

activations = {
    "ReLU": F.relu(x_range),
    "Sigmoid": torch.sigmoid(x_range),
    "Tanh": torch.tanh(x_range),
    "GELU (used in GPT)": F.gelu(x_range),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, vals) in zip(axes, activations.items()):
    ax.plot(x_range.numpy(), vals.numpy(), linewidth=2.5, color="#4C72B0")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(name, fontweight="bold")
    ax.set_xlim(-3, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle("Activation Functions — the non-linearities that make deep networks powerful",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Part 2 — The loss function, visualised

Training a network means finding the weights that produce the lowest possible loss. To understand what that means, let's **visualise the loss surface** — the landscape the optimiser has to navigate.

We'll use a toy 2D problem: a network with just **two weights** (so we can plot it). The loss surface is a hill-and-valley landscape. Training is the process of rolling downhill to the lowest point.

In [ ]:
# A simple 1-input → 1-output network with two parameters: weight w and bias b.
# We want the network to learn: output = 2.5 * input
# True relationship: y = 2.5 * x  (imagine x = revenue growth, y = stock move)

torch.manual_seed(0)
x_data = torch.linspace(-2, 2, 20)
y_data = 2.5 * x_data + torch.randn(20) * 0.3   # true line + small noise

def compute_loss(w_val, b_val):
    """Mean squared error loss for y_pred = w*x + b."""
    y_pred = w_val * x_data + b_val
    return ((y_pred - y_data) ** 2).mean().item()

# Compute loss across a grid of (w, b) values
w_range = np.linspace(-1, 5, 100)
b_range = np.linspace(-3, 3, 100)
W, B = np.meshgrid(w_range, b_range)
Z = np.array([[compute_loss(w, b) for w in w_range] for b in b_range])

# Plot the loss surface as a filled contour map
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: 2D contour plot
cs = axes[0].contourf(W, B, Z, levels=40, cmap="RdYlGn_r")
plt.colorbar(cs, ax=axes[0], label="Loss (MSE)")
axes[0].scatter([2.5], [0], color="white", s=200, zorder=5,
                edgecolors="black", linewidths=2, label="True minimum (w=2.5, b=0)")
axes[0].set_xlabel("Weight w", fontsize=11)
axes[0].set_ylabel("Bias b", fontsize=11)
axes[0].set_title("Loss Surface (top-down view)\nGreen = low loss, Red = high loss",
                  fontweight="bold")
axes[0].legend(loc="upper left", fontsize=9)

# Right: 3D surface
ax3d = fig.add_subplot(122, projection="3d")
ax3d.plot_surface(W, B, Z, cmap="RdYlGn_r", alpha=0.85, linewidth=0)
ax3d.set_xlabel("Weight w")
ax3d.set_ylabel("Bias b")
ax3d.set_zlabel("Loss")
ax3d.set_title("Loss Surface (3D view)", fontweight="bold")
ax3d.view_init(elev=30, azim=-120)

# Remove the flat subplot we added for 3D
fig.delaxes(axes[1])

plt.suptitle("The Loss Landscape — training is the process of finding the lowest point",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("The white dot marks the true minimum: w=2.5, b=0.")
print("Training starts somewhere random on this surface and navigates toward that minimum.")

---

## Part 3 — Gradient descent in action

The **gradient** at any point on the loss surface is a vector pointing in the direction of steepest *increase*. To minimise loss, we step in the *opposite* direction — this is gradient descent.

```
new_weight = old_weight - learning_rate × gradient
```

Let's watch this happen step by step on our 2D loss surface.

In [ ]:
# Run gradient descent manually so we can record every step

def run_gradient_descent(lr, start_w, start_b, num_steps=60):
    """
    Manually run gradient descent and record the path through (w, b) space.
    Returns lists of w values, b values, and loss values at each step.
    """
    w = torch.tensor(start_w, dtype=torch.float32, requires_grad=True)
    b = torch.tensor(start_b, dtype=torch.float32, requires_grad=True)
    
    path_w, path_b, path_loss = [start_w], [start_b], []
    
    for step in range(num_steps):
        # Forward: compute predictions and loss
        y_pred = w * x_data + b
        loss = ((y_pred - y_data) ** 2).mean()
        path_loss.append(loss.item())
        
        # Backward: compute gradients
        loss.backward()
        
        # Update weights manually (no optimiser — raw gradient descent)
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        
        # Zero gradients for next step
        w.grad.zero_()
        b.grad.zero_()
        
        path_w.append(w.item())
        path_b.append(b.item())
    
    return path_w, path_b, path_loss


# Three runs: a good learning rate, too small, too large
runs = {
    "Good LR (0.1)": run_gradient_descent(lr=0.1,  start_w=-0.5, start_b=2.5),
    "Too small (0.01)": run_gradient_descent(lr=0.01, start_w=-0.5, start_b=2.5),
    "Too large (0.8)": run_gradient_descent(lr=0.8,  start_w=-0.5, start_b=2.5),
}

colors = {"Good LR (0.1)": "#2196F3", "Too small (0.01)": "#FF9800", "Too large (0.8)": "#E53935"}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: path through weight space overlaid on loss contours
cs = axes[0].contourf(W, B, Z, levels=40, cmap="RdYlGn_r", alpha=0.7)
axes[0].scatter([2.5], [0], color="white", s=300, zorder=10,
                edgecolors="black", linewidths=2.5, label="Minimum", marker="*")

for label, (path_w, path_b, _) in runs.items():
    axes[0].plot(path_w, path_b, "-o", color=colors[label], markersize=3,
                 linewidth=2, label=label, alpha=0.9)
    axes[0].plot(path_w[0], path_b[0], "s", color=colors[label], markersize=10,
                 zorder=5)  # starting point

axes[0].set_xlabel("Weight w", fontsize=11)
axes[0].set_ylabel("Bias b", fontsize=11)
axes[0].set_title("Gradient Descent Paths through Loss Surface\n(squares = start, star = minimum)",
                  fontweight="bold")
axes[0].legend(loc="upper left", fontsize=9)
axes[0].set_xlim(-1, 5)
axes[0].set_ylim(-3, 3)

# Right: loss over steps
for label, (_, _, path_loss) in runs.items():
    axes[1].plot(path_loss, color=colors[label], linewidth=2.5, label=label)

axes[1].set_xlabel("Training step", fontsize=11)
axes[1].set_ylabel("Loss", fontsize=11)
axes[1].set_title("Loss vs Training Steps\n(how quickly does each learning rate converge?)",
                  fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale("log")  # log scale makes the differences clearer

plt.suptitle("Gradient Descent in Action — the learning rate controls step size",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Blue  (LR=0.1)  — converges smoothly and quickly to the minimum")
print("Orange (LR=0.01) — moves in the right direction but takes many more steps")
print("Red   (LR=0.8)  — overshoots, bounces around — may diverge entirely")
print()
print("Choosing the learning rate is one of the most important decisions in training.")
print("Too small = slow. Too large = unstable. The right value is found by experiment.")

In [ ]:
# Let's also visualise the gradient vectors themselves at a few points on the surface.
# Each arrow shows: direction = gradient, length = gradient magnitude.
# Gradient descent moves in the OPPOSITE direction to these arrows.

fig, ax = plt.subplots(figsize=(8, 6))

cs = ax.contourf(W, B, Z, levels=30, cmap="RdYlGn_r", alpha=0.75)
plt.colorbar(cs, ax=ax, label="Loss")

# Compute and plot gradient vectors at a grid of points
sample_w = np.linspace(-0.5, 4.5, 10)
sample_b = np.linspace(-2.5, 2.5, 10)

for sw in sample_w:
    for sb in sample_b:
        w_t = torch.tensor(sw, dtype=torch.float32, requires_grad=True)
        b_t = torch.tensor(sb, dtype=torch.float32, requires_grad=True)
        y_pred = w_t * x_data + b_t
        loss = ((y_pred - y_data) ** 2).mean()
        loss.backward()
        
        gw = w_t.grad.item()
        gb = b_t.grad.item()
        magnitude = np.sqrt(gw**2 + gb**2)
        
        if magnitude > 0:
            # Normalise arrow length so they're readable, scale by magnitude
            scale = 0.25 * min(1.0, 3.0 / magnitude)
            ax.annotate("", xy=(sw + gw * scale, sb + gb * scale),
                        xytext=(sw, sb),
                        arrowprops=dict(arrowstyle="->", color="white", lw=1.2, alpha=0.8))

ax.scatter([2.5], [0], color="yellow", s=300, zorder=10,
           edgecolors="black", linewidths=2, marker="*", label="Minimum (w=2.5, b=0)")
ax.set_xlabel("Weight w", fontsize=11)
ax.set_ylabel("Bias b", fontsize=11)
ax.set_title("Gradient Vectors on the Loss Surface\n"
             "Arrows point uphill — gradient descent moves opposite to them",
             fontweight="bold")
ax.legend(loc="upper left")
ax.set_xlim(-1, 5)
ax.set_ylim(-3, 3)
plt.tight_layout()
plt.show()

print("White arrows = gradient direction (steepest increase in loss).")
print("Gradient descent moves AGAINST these arrows — always downhill.")
print("Arrows near the minimum are short (shallow gradient), far away they're long (steep gradient).")

---

## Part 4 — A feedforward neural network on financial data

Now we scale up: many neurons, multiple layers, a real classification task. We'll also record gradient magnitudes during training to see how learning evolves layer by layer.

In [ ]:
# Financial headline sentiment dataset
# Features: [positive_word_count, negative_word_count, revenue_mentioned, beat_mentioned, miss_mentioned]
# Label: 1 = bullish, 0 = bearish

raw_data = [
    ([3, 0, 1, 1, 0], 1, "Revenue beats estimates, strong quarterly growth"),
    ([0, 3, 1, 0, 1], 0, "Revenue misses estimates, weak demand outlook"),
    ([2, 1, 0, 1, 0], 1, "Earnings beat expectations, raises guidance"),
    ([1, 2, 1, 0, 1], 0, "Sales miss forecasts, lowering annual guidance"),
    ([4, 0, 1, 1, 0], 1, "Record revenue, strong margins, beat on all metrics"),
    ([0, 4, 0, 0, 1], 0, "Weak results, significant miss, cutting costs"),
    ([2, 0, 1, 0, 0], 1, "Revenue growth accelerates, new products launch well"),
    ([0, 2, 0, 0, 1], 0, "Disappointing results, restructuring charges announced"),
    ([3, 1, 1, 1, 0], 1, "Strong earnings beat, raised dividend, revenue up"),
    ([1, 3, 1, 0, 1], 0, "Revenue miss, margin pressure, cuts workforce"),
    ([2, 0, 0, 1, 0], 1, "Beat on EPS, positive outlook, shares rise"),
    ([0, 2, 1, 0, 0], 0, "Revenue declines, market share losses continue"),
    ([3, 0, 0, 1, 0], 1, "Impressive growth, analyst upgrades, strong buy signals"),
    ([1, 2, 0, 0, 1], 0, "Miss on key metrics, regulatory headwinds growing"),
    ([2, 1, 1, 0, 0], 1, "Revenue in line, margins expand, positive signals"),
    ([0, 3, 0, 0, 1], 0, "Significant miss, poor guidance, sell-off continues"),
]

X = torch.tensor([d[0] for d in raw_data], dtype=torch.float32)
y = torch.tensor([d[1] for d in raw_data], dtype=torch.float32)

print(f"Dataset: {len(raw_data)} financial headlines")
print(f"Features: [positive_words, negative_words, revenue_mentioned, beat_mentioned, miss_mentioned]")

In [ ]:
class FinancialSentimentNet(nn.Module):
    """Input(5) → Linear(16) → ReLU → Linear(8) → ReLU → Linear(1) → Sigmoid"""
    
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(5, 16)
        self.layer2 = nn.Linear(16, 8)
        self.layer3 = nn.Linear(8, 1)
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = torch.sigmoid(self.layer3(x))
        return x.squeeze()


torch.manual_seed(42)
model = FinancialSentimentNet()

total_params = sum(p.numel() for p in model.parameters())
print(f"Network: {total_params} parameters")
print(f"  layer1: 5×16 + 16 = {5*16+16}")
print(f"  layer2: 16×8 + 8  = {16*8+8}")
print(f"  layer3: 8×1 + 1   = {8*1+1}")

In [ ]:
# Training loop — now we also record gradient magnitudes per layer
# This lets us see how actively each layer is updating during training

loss_fn  = nn.BCELoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.01)

loss_history     = []
accuracy_history = []
grad_history     = {"layer1": [], "layer2": [], "layer3": []}  # gradient magnitudes per layer

num_epochs = 200

for epoch in range(num_epochs):
    predictions = model(X)
    loss = loss_fn(predictions, y)
    
    optimiser.zero_grad()
    loss.backward()
    
    # Record the L2 norm of gradients for each layer's weights
    # L2 norm = sqrt(sum of squared gradients) — a measure of how large the update is
    for layer_name in ["layer1", "layer2", "layer3"]:
        layer = getattr(model, layer_name)
        grad_norm = layer.weight.grad.norm().item()
        grad_history[layer_name].append(grad_norm)
    
    optimiser.step()
    
    accuracy = ((predictions > 0.5) == y.bool()).float().mean().item()
    loss_history.append(loss.item())
    accuracy_history.append(accuracy)

print(f"Training complete.")
print(f"Final loss: {loss_history[-1]:.4f}  |  Final accuracy: {accuracy_history[-1]:.0%}")

In [ ]:
# ── Plot 1: Loss and accuracy together ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
axes[0].plot(loss_history, color="#E53935", linewidth=2.5)
axes[0].fill_between(range(len(loss_history)), loss_history, alpha=0.1, color="#E53935")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Loss (Binary Cross-Entropy)", fontsize=11)
axes[0].set_title("Training Loss", fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Mark the point where loss halved
start_loss = loss_history[0]
for i, l in enumerate(loss_history):
    if l < start_loss / 2:
        axes[0].axvline(x=i, color="gray", linestyle="--", alpha=0.7)
        axes[0].text(i + 2, start_loss * 0.7, f"Loss halved\n@ epoch {i}",
                    fontsize=8, color="gray")
        break

# Accuracy curve
axes[1].plot(accuracy_history, color="#43A047", linewidth=2.5)
axes[1].fill_between(range(len(accuracy_history)), accuracy_history, alpha=0.1, color="#43A047")
axes[1].axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="100% accuracy")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Accuracy", fontsize=11)
axes[1].set_title("Training Accuracy", fontweight="bold")
axes[1].set_ylim(0, 1.05)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=9)

plt.suptitle("As loss falls, accuracy rises — the network is learning the financial sentiment task",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Gradient magnitudes per layer ────────────────────────────────────
# This shows how much each layer's weights are changing at each step.
# Early in training: large gradients — the network is making big adjustments.
# Late in training: small gradients — converging toward a solution.

fig, ax = plt.subplots(figsize=(12, 4))

layer_colors = {"layer1": "#4C72B0", "layer2": "#55A868", "layer3": "#C44E52"}
layer_labels = {
    "layer1": "Layer 1 (5→16, closest to input)",
    "layer2": "Layer 2 (16→8, middle)",
    "layer3": "Layer 3 (8→1, output layer)",
}

for layer_name, grads in grad_history.items():
    # Smooth with a rolling average to reduce noise
    smoothed = np.convolve(grads, np.ones(10)/10, mode="valid")
    ax.plot(smoothed, color=layer_colors[layer_name],
            linewidth=2, label=layer_labels[layer_name])

ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Gradient L2 Norm", fontsize=11)
ax.set_title("Gradient Magnitudes per Layer during Training\n"
             "How much is each layer updating at each step?",
             fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Large gradients early = network making big corrections from a bad starting point.")
print("Gradients shrink as the network approaches the minimum — smaller corrections needed.")
print()
print("If gradients go to zero too early: 'vanishing gradient' problem.")
print("If gradients explode: 'exploding gradient' problem — this is why we use gradient clipping.")

In [ ]:
# ── Plot 3: How confident is the network over time? ──────────────────────────
# We replay training and record the model's prediction confidence on two examples:
# one clearly bullish, one clearly bearish.

torch.manual_seed(42)
model2 = FinancialSentimentNet()
optimiser2 = torch.optim.Adam(model2.parameters(), lr=0.01)

# Two fixed test examples to track
# Example 0: [4, 0, 1, 1, 0] label=1 — strongly bullish
# Example 5: [0, 4, 0, 0, 1] label=0 — strongly bearish
conf_bullish  = []
conf_bearish  = []

for epoch in range(200):
    preds = model2(X)
    loss2 = loss_fn(preds, y)
    optimiser2.zero_grad()
    loss2.backward()
    optimiser2.step()
    
    with torch.no_grad():
        p = model2(X)
    conf_bullish.append(p[4].item())   # strongly bullish example
    conf_bearish.append(p[5].item())   # strongly bearish example

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(conf_bullish, color="#43A047", linewidth=2.5,
        label="Bullish example: 'Record revenue, strong margins' (true label = 1)")
ax.plot(conf_bearish, color="#E53935", linewidth=2.5,
        label="Bearish example: 'Weak results, significant miss' (true label = 0)")

ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Decision boundary (0.5)")
ax.fill_between(range(200), 0.5, 1.0, alpha=0.05, color="green")
ax.fill_between(range(200), 0.0, 0.5, alpha=0.05, color="red")

ax.text(180, 0.92, "Bullish region", color="green", fontsize=9, alpha=0.7)
ax.text(180, 0.05, "Bearish region", color="red",   fontsize=9, alpha=0.7)

ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Predicted probability", fontsize=11)
ax.set_title("Model Confidence over Training\n"
             "Network becomes more certain as it learns the task",
             fontweight="bold")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Early in training: both predictions hover near 0.5 — the network is guessing.")
print("As training progresses: the network pushes bullish toward 1.0 and bearish toward 0.0.")
print("This is what 'learning' looks like from the inside.")

In [ ]:
# Final predictions table
with torch.no_grad():
    final_preds = model(X)

print(f"{'Headline':55s} {'True':>6} {'Pred':>7} {'✓/✗':>5}")
print("-" * 78)
for i, (_, label, headline) in enumerate(raw_data):
    pred = final_preds[i].item()
    correct = "✓" if (pred > 0.5) == bool(label) else "✗"
    print(f"{headline[:54]:55s} {label:>6}  {pred:>6.2f}  {correct:>5}")

---

## Part 5 — A tiny language model from scratch

Language models are neural networks trained to **predict the next character (or word) in a sequence**. At sufficient scale, this single task forces a model to learn grammar, facts, and reasoning.

We'll build a character-level language model and watch its loss and gradient behaviour during training.

In [ ]:
text = """
revenue grew strongly in the fiscal year driven by product demand.
earnings per share beat analyst estimates by a significant margin.
the company reported record net income for the quarter.
operating margins expanded as cost efficiencies were realised.
revenue declined due to weaker consumer demand and supply constraints.
the board declared a quarterly dividend and announced a share buyback.
net interest income rose as interest rates increased through the year.
cloud services revenue grew strongly driven by enterprise adoption.
the company guided for revenue growth in the range of ten to fifteen percent.
gross margin improved as the product mix shifted toward higher margin services.
capital expenditure increased as the company invested in data centre expansion.
free cash flow generation remained strong supporting continued shareholder returns.
the acquisition was completed and is expected to be accretive to earnings.
currency headwinds reduced reported revenue growth by approximately two percent.
research and development spending increased to drive future product innovation.
""".strip().lower()

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
encoded = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

CONTEXT_LEN = 20
inputs_lm  = torch.stack([encoded[i:i+CONTEXT_LEN] for i in range(len(encoded)-CONTEXT_LEN)])
targets_lm = torch.stack([encoded[i+CONTEXT_LEN]   for i in range(len(encoded)-CONTEXT_LEN)])

print(f"Corpus: {len(text)} chars, vocab: {vocab_size}, training examples: {len(inputs_lm)}")

In [ ]:
EMBED_DIM  = 16
HIDDEN_DIM = 128

class TinyLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.fc1 = nn.Linear(CONTEXT_LEN * EMBED_DIM, HIDDEN_DIM)
        self.fc2 = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.fc3 = nn.Linear(HIDDEN_DIM, vocab_size)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.embedding(x).view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        return self.fc3(x)

torch.manual_seed(42)
lm = TinyLanguageModel().to(device)
lm_loss_fn   = nn.CrossEntropyLoss()
lm_optimiser = torch.optim.Adam(lm.parameters(), lr=3e-3)

X_lm = inputs_lm.to(device)
y_lm = targets_lm.to(device)

lm_loss_history  = []
lm_grad_history  = {"embedding": [], "fc1": [], "fc2": [], "fc3": []}
BATCH_SIZE = 64

print(f"Training tiny language model ({sum(p.numel() for p in lm.parameters()):,} params)...")

for epoch in range(300):
    idx = torch.randperm(len(X_lm))[:BATCH_SIZE]
    logits = lm(X_lm[idx])
    loss   = lm_loss_fn(logits, y_lm[idx])
    
    lm_optimiser.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(lm.parameters(), max_norm=1.0)
    
    lm_grad_history["embedding"].append(lm.embedding.weight.grad.norm().item())
    for layer in ["fc1", "fc2", "fc3"]:
        lm_grad_history[layer].append(getattr(lm, layer).weight.grad.norm().item())
    
    lm_optimiser.step()
    lm_loss_history.append(loss.item())

print(f"Final loss: {lm_loss_history[-1]:.4f}")

In [ ]:
# Plot loss + gradient magnitudes for the language model
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Smooth loss history
smooth_loss = np.convolve(lm_loss_history, np.ones(15)/15, mode="valid")
axes[0].plot(lm_loss_history, color="#90CAF9", linewidth=1, alpha=0.4, label="Raw")
axes[0].plot(smooth_loss, color="#1565C0", linewidth=2.5, label="Smoothed")
axes[0].set_xlabel("Training step", fontsize=11)
axes[0].set_ylabel("Cross-entropy loss", fontsize=11)
axes[0].set_title("Language Model Training Loss\n"
                  "Lower = better next-character predictions",
                  fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gradient magnitudes — 4 layers
lm_colors = {"embedding": "#7B1FA2", "fc1": "#4C72B0", "fc2": "#55A868", "fc3": "#C44E52"}
lm_labels = {
    "embedding": "Embedding (char → vector)",
    "fc1": "FC1 (context → hidden)",
    "fc2": "FC2 (hidden → hidden)",
    "fc3": "FC3 (hidden → vocab)",
}
for layer, grads in lm_grad_history.items():
    smoothed = np.convolve(grads, np.ones(10)/10, mode="valid")
    axes[1].plot(smoothed, color=lm_colors[layer], linewidth=2, label=lm_labels[layer])

axes[1].set_xlabel("Training step", fontsize=11)
axes[1].set_ylabel("Gradient L2 Norm", fontsize=11)
axes[1].set_title("Gradient Magnitudes per Layer\n"
                  "Each layer learns at a different rate",
                  fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Language Model Training Dynamics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Generate text from the trained model

def generate_text(model, seed_text, num_chars=150, temperature=0.7):
    model.eval()
    seed_text = seed_text.lower()
    if len(seed_text) < CONTEXT_LEN:
        seed_text = seed_text.rjust(CONTEXT_LEN)
    context = [char_to_idx.get(c, 0) for c in seed_text[-CONTEXT_LEN:]]
    generated = seed_text
    with torch.no_grad():
        for _ in range(num_chars):
            x = torch.tensor([context], dtype=torch.long).to(device)
            logits = model(x) / temperature
            probs  = F.softmax(logits, dim=-1)
            next_id   = torch.multinomial(probs, num_samples=1).item()
            generated += idx_to_char[next_id]
            context    = context[1:] + [next_id]
    return generated

for seed in ["revenue grew", "the company reported", "earnings per share"]:
    print(f"Seed: '{seed}'")
    print("-" * 50)
    print(generate_text(lm, seed, num_chars=120, temperature=0.7))
    print()

---

## What we visualised

| Chart | What it shows |
|-------|---------------|
| Loss surface (contour + 3D) | The landscape the optimiser navigates — valleys = good weights, peaks = bad |
| Gradient vectors | Direction and steepness at each point — descent moves opposite to these |
| Gradient descent paths | How learning rate controls convergence speed and stability |
| Training loss + accuracy | The network improving over time on our financial task |
| Gradient magnitudes per layer | How actively each layer is updating — large early, small when converged |
| Model confidence over training | Individual predictions moving from uncertainty (0.5) toward conviction |
| LM training dynamics | The same curves you see in Notebook 04's fine-tuning run |

---

## Connecting to the full workflow

Everything you've seen here — loss surface, gradient descent, training curves — is happening at exactly the same level inside TinyLlama when we fine-tune it in Notebook 04. The only difference is scale: 1.1 billion parameters instead of a few hundred, and a vocabulary of 32,000 tokens instead of 30 characters.

The training loop in Notebook 04 is:
```python
predictions = model(X)      # forward pass
loss = loss_fn(predictions) # compute loss
loss.backward()             # backward pass — gradients
optimiser.step()            # update weights
```
Exactly what we wrote here, wrapped in the HuggingFace `Trainer` class.

---

▶ **Next: [Notebook 02 — Data Preparation](02_data_preparation.ipynb)**